# Day 09 — Reranking and context selection: a second look at the candidates, measured

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-09/LEARN.md`](../docs/day-09/LEARN.md).*

Day 08's weighted fusion puts a labelled chunk in the top 3 for every answerable query on Day 07's labelled set. Two problems are left: the **order** inside the top 3 isn't always right, and five unanswerable test questions still get a confident, cited, wrong answer, because the chunks retrieved for them look relevant without answering them.

Both retrievers score a chunk without ever reading it next to the question: BM25 counts shared words, and dense retrieval compares two vectors made separately. A **cross-encoder reranker** reads the question and the chunk together, in one input, and outputs one relevance score. It is too slow to run on every chunk, so it reranks a short list of candidates that the fast retrievers found. This notebook builds that from scratch and measures it:

1. **A cross-encoder**, from its two files to one score per (question, chunk) pair.
2. **Rerank the hybrid's candidates** (Day 08's weighted fusion, and RRF for comparison), on the same 81 queries, query by query against Day 08's accepted baseline.
3. **What moved, and why**; what the second look **costs**.
4. **Can it tell "close" from "answers"?** The five "similar but wrong" questions.
5. **Context selection**: duplicates, source diversity and the budget.
6. **Permissions**: what a reranker does with a chunk that shouldn't be there.

**Every setting was committed before anything scored the test split.** [`evals/preregistration/day-09.md`](../evals/preregistration/day-09.md) holds the rules (each reads only the dev split), the values they gave, what would be reported, and what each decision needs. It was committed before this notebook first ran. The notebook recomputes every rule on dev and stops if it gets a different value. The v1 test split is no longer clean (Days 07 and 08 saw its results while exploring), so section 9 says what today's test numbers can and can't show.

Everything runs locally: no API key, no new packages, one new model download (91 MB), about 1–2 minutes.

## 0. Setup

No new packages: the reranker runs on the ones Day 04 installed (onnxruntime, tokenizers and huggingface-hub). The shared code in [`src/rag_from_scratch/`](../src/rag_from_scratch/) gains **Day 08's fusion** (`fusion.py`: `rrf()`, `min_max()`, `weighted()` and `rrf_margin()`), because today reranks its output. `tests/test_shared_code.py` checks that the module returns exactly what the Day 08 notebook returns.

In [1]:
import hashlib
import json
import math
import platform
import statistics
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
# huggingface_hub's progress bars warn when Jupyter widgets aren't installed; they still work as plain text.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import numpy as np
    import onnxruntime as ort
    import tokenizers
    from huggingface_hub import hf_hub_download
    from tokenizers import Tokenizer
except ImportError as error:
    raise ImportError("Day 09 uses Day 04's packages (numpy, onnxruntime, tokenizers, huggingface-hub). From the "
                      "repository root run: python -m pip install -r requirements.txt") from error


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))           # import the shared code without installing it

from rag_from_scratch.answering import (DIRECTORY, MAX_CONTEXT_WORDS, Pipeline, assemble,  # Day 06's pipeline parts
                                        count_words, unknown_names)
from rag_from_scratch.bm25 import BM25Index                              # Day 01's BM25
from rag_from_scratch.chunking import section_chunks                     # Day 03's chunker
from rag_from_scratch.corpus import corpus_after                         # Day 05's "documents changed" helper
from rag_from_scratch.embeddings import Embedder, file_sha256            # Day 04's model
from rag_from_scratch.evaluation import (TIE, compare, fingerprint, first_relevant, hit, load_queries,  # Day 07's
                                         measure, sign_test, snapshot, tie_margin, top_score, validate,   # harness
                                         wilson)
from rag_from_scratch.fusion import ALPHA, RRF_K, rrf, rrf_margin, weighted   # Day 08's fusion (new in src/)
from rag_from_scratch.index import VectorIndex, sync                     # Day 05's index
from rag_from_scratch.ingest import ingest                               # Day 02's pipeline

print(f"Python {platform.python_version()} | numpy {np.__version__} | onnxruntime {ort.__version__} | "
      f"tokenizers {tokenizers.__version__}")

Python 3.13.17 | numpy 2.4.6 | onnxruntime 1.31.0 | tokenizers 0.23.3


In [2]:
TOP_K = 3                # Day 07's definition: an ERROR is an answerable query with no labelled chunk in its top 3
K_VALUES = (1, 3, 5, 10)
EVALS = REPO_ROOT / "evals"
QUESTIONS_FILE = EVALS / "questions-v1.jsonl"
MANIFEST_FILE = EVALS / "manifest.json"
DAY08_BASELINE = EVALS / "baselines" / "day-08-retrieval.json"   # Day 08's accepted run: today's reference
BASELINE_FILE = EVALS / "baselines" / "day-09-retrieval.json"    # today's accepted run, committed and reviewed
PREREGISTRATION = EVALS / "preregistration" / "day-09.md"        # today's rules, committed before any test-split run
OUT_DIR = REPO_ROOT / "outputs" / "day-09"                       # this run's results; outputs/ is git-ignored

# The settings evals/preregistration/day-09.md commits to. Sections 3 and 6 recompute each one with its rule on the
# dev split and stop if the rule gives anything else.
PREREGISTERED = {"depth": {"weighted": 3, "RRF": 20}, "gate": -3.6}
DEPTHS = (3, 5, 10, 20, 44)      # how many of the hybrid's candidates the reranker may read (44 = every chunk)
DENSE_GATE = 0.62                # Day 07's measured gate on the dense top score (evals/baselines/day-07-retrieval.json)

# The reranker: an ONNX export of cross-encoder/ms-marco-MiniLM-L6-v2, pinned to a commit, every file checked
RERANKER = {
    "repo": "Xenova/ms-marco-MiniLM-L-6-v2",                   # https://huggingface.co/Xenova/ms-marco-MiniLM-L-6-v2
    "revision": "a09144355adeed5f58c8ed011d209bf8ee5a1fec",  # a commit, not "main": the files can't change
    "converted_from": "cross-encoder/ms-marco-MiniLM-L6-v2",  # https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2
    "licence": "Apache-2.0 (the original model's; the ONNX repository states none of its own)",
    "files": {                                                 # path in the repository: (SHA-256, bytes)
        "onnx/model.onnx": ("c623d0bcb99f4622beb413eaef00cfbe5db20df9f1dd982da4b4f26022881870", 90_992_115),
        "tokenizer.json": ("d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66", 711_396),
    },
}
RERANKER_DIR = REPO_ROOT / "models" / "ms-marco-MiniLM-L-6-v2" / RERANKER["revision"][:12]   # models/ is git-ignored
MAX_TOKENS = 512         # the model's input limit: question and chunk together

prereg = PREREGISTRATION.read_text(encoding="utf-8")
assert "**N = 3**" in prereg and "**N = 20**" in prereg and "**the gate is −3.6**" in prereg, \
    "PREREGISTERED must match evals/preregistration/day-09.md"

## 1. The same queries, the same snapshot, Day 08's four retrievers

A comparison is only fair if everything except the new step stays the same. So, before building anything, this section loads Day 07's labels, checks the documents against their snapshot, rebuilds Day 08's four retrievers (BM25, dense, RRF and the weighted fusion) and checks that they **reproduce Day 08's accepted baseline**, query by query.

In [3]:
manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
queries = load_queries(QUESTIONS_FILE)
by_qid = {q.id: q for q in queries}
documents, _ = ingest(REPO_ROOT)
chunks = [c for d in documents for c in section_chunks(d)]
by_cid = {c.chunk_id: c for c in chunks}
problems = validate(queries, chunks, manifest)
file_sha256_ = hashlib.sha256(QUESTIONS_FILE.read_bytes()).hexdigest()
if file_sha256_ != manifest["questions_sha256"]:
    problems.insert(0, f"{QUESTIONS_FILE.name} changed but the manifest didn't")
print(f"{len(queries)} labelled queries (version {manifest['version']}) | documents: {len(chunks)} chunks, snapshot "
      f"{fingerprint(snapshot(chunks))[:12]}... | labels written on {manifest['snapshot']['fingerprint'][:12]}...")
print(f"label problems: {len(problems)}" + "".join(f"\n  {p}" for p in problems))
assert not problems, "The labels don't match the documents: run the Day 07 notebook and review them first."

embedder = Embedder(REPO_ROOT)     # Day 04's model: downloaded once (133 MB), SHA-256 checked on every run
index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
sync(index, chunks, embedder.embed)   # the snapshot's chunks, embedded now (Day 05's sync into an empty index)
rag = Pipeline(index, embedder)       # Day 06's retrieve(): as the user, permission filter before the ranking
bm25 = {user: BM25Index({c.chunk_id: c.text for c in chunks if set(c.allowed_groups) & groups})
        for user, groups in DIRECTORY.items()}


def bm25_search(q, k=None):
    # Day 07's BM25: the user's own index. Only chunks that share a word with the query come back.
    return [(h.id, h.score) for h in bm25[q.user].search(q.question, k or len(chunks))]


def dense_search(q, k=None):
    # Day 07's dense retrieval: every chunk the user may read, ranked by cosine similarity
    return [(r.id, r.score) for r in rag.retrieve(q.user, q.question, k or len(chunks))]


# Every query's FULL ranking from each retriever: (chunk_id, score), best first. Day 08's two hybrids come from
# rag_from_scratch.fusion, a tested copy of the Day 08 notebook's code.
rankings = {"BM25": {q.id: bm25_search(q) for q in queries}, "dense": {q.id: dense_search(q) for q in queries}}
rankings["RRF"] = {q.id: rrf([rankings["BM25"][q.id], rankings["dense"][q.id]]) for q in queries}
rankings["weighted"] = {q.id: weighted(rankings["BM25"][q.id], rankings["dense"][q.id], ALPHA) for q in queries}
HYBRIDS = ("weighted", "RRF")
SELECTIONS = {"questions": [q for q in queries if q.kind == "question" and q.answerable],   # unanswerable ones
              "lookups": [q for q in queries if q.kind == "lookup"]}                        # have no label to find
print(f"{len(SELECTIONS['questions'])} answerable questions (dev {sum(q.split == 'dev' for q in SELECTIONS['questions'])},"
      f" test {sum(q.split == 'test' for q in SELECTIONS['questions'])}), {len(SELECTIONS['lookups'])} lookups (all dev) | "
      f"hybrids: weighted (alpha {ALPHA}) and RRF (k {RRF_K})")

81 labelled queries (version 1) | documents: 44 chunks, snapshot 3f8344d00b8b... | labels written on 3f8344d00b8b...
label problems: 0
tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified
onnx/model.onnx  133,093,490 bytes   SHA-256 828e1496d7fa... verified


31 answerable questions (dev 16, test 15), 30 lookups (all dev) | hybrids: weighted (alpha 0.7) and RRF (k 60)


In [4]:
day08 = json.loads(DAY08_BASELINE.read_text(encoding="utf-8"))
DATASET = {"questions": QUESTIONS_FILE.relative_to(REPO_ROOT).as_posix(), "version": manifest["version"],
           "sha256": file_sha256_, "snapshot": manifest["snapshot"]["fingerprint"]}
SETTINGS = {   # Day 08's four retrievers exactly as Day 08 described them, so compare() checks the settings too
    "BM25": {"code": "rag_from_scratch.bm25 (Day 01)", "k1": bm25["alice"].k1, "b": bm25["alice"].b,
             "indexed_text": "chunk text with its Title > Section header",
             "permissions": "one index of each user's readable chunks (filter before the ranking)"},
    "dense": {"code": "rag_from_scratch.answering.Pipeline.retrieve (Day 06)", "model": embedder.model["repo"],
              "revision": embedder.revision, "query_instruction": True,
              "indexed_text": "chunk text with its Title > Section header",
              "permissions": "metadata filter before the ranking"},
    "RRF": {"code": "rrf() in the Day 08 notebook", "fuses": ["BM25", "dense"], "k": RRF_K,
            "inputs": "full rankings, each already permission-filtered", "ties": "chunk id"},
    "weighted": {"code": "weighted() in the Day 08 notebook", "fuses": ["BM25", "dense"], "alpha_dense": ALPHA,
                 "normalisation": "min-max per retriever and query; 0 from a retriever that didn't return the chunk",
                 "alpha_rule": "most dev queries in the top 3, then the higher dev MRR@10, then closest to 0.5",
                 "inputs": "full rankings, each already permission-filtered", "ties": "chunk id"}}
DAY08_NAMES = ["BM25", "dense", "RRF", "weighted"]
RERANKED = {}            # filled in section 3: {"weighted+rerank": "weighted", "RRF+rerank": "RRF"}


def entry(name: str, q) -> dict:
    # One query, one retriever, in the baseline's format: the rank of the first labelled chunk, how close that is to a
    # tie, the top 3 and the best score. RRF's margin is in its inputs (Day 08); a reranked one is explained in section 10.
    ranked = rankings[name][q.id]
    if not q.answerable:
        margin = None
    elif name == "RRF":
        margin = rrf_margin([rankings["BM25"][q.id], rankings["dense"][q.id]], q.answer_chunks)
    elif name in RERANKED:
        margin = rerank_margin(name, q)
    else:
        margin = tie_margin(ranked, q.answer_chunks)
    return {"first_relevant": first_relevant(ranked, q.answer_chunks) if q.answerable else None,
            "margin": margin, "top3": [cid for cid, _ in ranked[:TOP_K]], "top_score": top_score(ranked)}


def metrics_of(names) -> dict:
    return {f"{kind}, {name}": {k: round(v, 4) if isinstance(v, float) else v
                                for k, v in measure(rankings[name], selection).items()}
            for kind, selection in SELECTIONS.items() for name in names}


# Day 08's accepted file, with today's settings, ranks and metrics for its four retrievers in place of its own
reproduced = {**day08, "dataset": DATASET, "retrievers": {name: SETTINGS[name] for name in DAY08_NAMES},
              "queries": {q.id: {name: entry(name, q) for name in DAY08_NAMES} for q in queries},
              "metrics": metrics_of(DAY08_NAMES)}
changes08, ties08 = compare(reproduced, day08)
same_top3 = all(reproduced["queries"][qid][n]["top3"] == day08["queries"][qid][n]["top3"] for qid in by_qid
                for n in DAY08_NAMES)
print(f"BM25, dense, RRF and weighted against {DAY08_BASELINE.relative_to(REPO_ROOT).as_posix()}: "
      + (f"{len(changes08)} changes" + "".join(f"\n  {c}" for c in changes08) if changes08 else
         f"no changes ({len(queries)} queries x 4 retrievers: every rank, best score and metric; top 3 lists "
         f"{'identical' if same_top3 else 'DIFFERENT'})"))
for t in ties08:
    print(f"  near tie, not a change: {t}")

BM25, dense, RRF and weighted against evals/baselines/day-08-retrieval.json: no changes (81 queries x 4 retrievers: every rank, best score and metric; top 3 lists identical)


Same labels, same snapshot, same four rankings: every difference below comes from the reranker.

## 2. A cross-encoder, from scratch

### 2.1 The model and its files

The reranker is **`cross-encoder/ms-marco-MiniLM-L6-v2`**: 22.7M parameters (6 layers), trained by the sentence-transformers project to judge whether a passage answers a search query. It was trained on **MS MARCO**, a large set of real Bing questions with passages that answer them, and its licence is Apache-2.0.

The notebook downloads an **ONNX export** of it (`Xenova/ms-marco-MiniLM-L-6-v2`), pinned to a commit, with every file's SHA-256 checked before it is loaded, exactly as Day 04 did for the embedding model. ONNX is a data format, so nothing in the file can run code when it loads, and onnxruntime is already installed. (The original repository now has its own `onnx/model.onnx` too, a different export; exercise 2 swaps it in.)

Two licence notes, checked on the model pages: the ONNX repository states no licence of its own (it is a format conversion of the Apache-2.0 model), and MS MARCO's terms say the dataset is for non-commercial research. Whether a dataset's terms bind a model trained on it is a legal question this notebook can't answer: check it before a commercial deployment. LEARN.md, section 4, has the details.

In [5]:
def reranker_file(name: str) -> Path:
    # Download one file of the pinned revision if it isn't here yet, then check its SHA-256 every time (as on Day 04).
    expected_sha256, expected_size = RERANKER["files"][name]
    local = RERANKER_DIR / name
    shown = local.relative_to(REPO_ROOT).as_posix()
    if not local.is_file():   # only the first run touches the network
        try:
            hf_hub_download(RERANKER["repo"], name, revision=RERANKER["revision"], local_dir=RERANKER_DIR)
        except Exception as error:   # no network, a proxy, a blocked host...: say how to fix it
            raise RuntimeError(f"Could not download {name} ({expected_size / 1e6:.1f} MB) from {RERANKER['repo']}: "
                               f"{error}\nBehind a proxy? Set HTTPS_PROXY. Hugging Face blocked? Set HF_ENDPOINT to a "
                               f"mirror, or download the file yourself and save it as {shown}") from error
    actual = file_sha256(local)
    if actual != expected_sha256:
        raise ValueError(f"{shown} has SHA-256 {actual}, expected {expected_sha256}. "
                         "Delete it and run again. Never load a model file you can't verify.")
    print(f"{name:<16} {local.stat().st_size:>11,} bytes   SHA-256 {actual[:12]}... verified")
    return local


pair_tokenizer = Tokenizer.from_file(str(reranker_file("tokenizer.json")))
pair_tokenizer.enable_truncation(max_length=MAX_TOKENS)   # question + chunk together, cut at the model's limit
pair_tokenizer.enable_padding()                            # in a batch, pad shorter pairs with [PAD]
reranker = ort.InferenceSession(str(reranker_file("onnx/model.onnx")), providers=["CPUExecutionProvider"])
inputs_, outputs_ = reranker.get_inputs(), reranker.get_outputs()
print(f"inputs: {', '.join(i.name for i in inputs_)} | output: {outputs_[0].name}, shape {outputs_[0].shape}: "
      "one number per (question, chunk) pair")
same = file_sha256(RERANKER_DIR / "tokenizer.json") == file_sha256(embedder.model_dir / "tokenizer.json")
print(f"tokenizer.json is {'byte for byte the same file as' if same else 'NOT the same file as'} Day 04's embedding "
      "model's (both use BERT's uncased vocabulary)")

tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified
onnx/model.onnx   90,992,115 bytes   SHA-256 c623d0bcb99f... verified


inputs: input_ids, attention_mask, token_type_ids | output: logits, shape ['batch_size', 1]: one number per (question, chunk) pair
tokenizer.json is byte for byte the same file as Day 04's embedding model's (both use BERT's uncased vocabulary)


### 2.2 One input: the question and the chunk together

A bi-encoder (Day 04's model) turns the question into one vector and each chunk into another, separately, and compares the two. A cross-encoder puts both texts in **one input**: `[CLS] question [SEP] chunk [SEP]`. A second row of numbers, the segment ids, says which tokens belong to which text. Every layer can then compare every question token with every chunk token.

In [6]:
DADS = by_qid["d01-01"]                                              # "How much time off do new dads get?"
ENTITLEMENT = by_cid["parental-leave-policy#leave-entitlement"]      # its answer
encoding = pair_tokenizer.encode(DADS.question, ENTITLEMENT.text)
segments = [[t for t, s in zip(encoding.tokens, encoding.type_ids) if s == part] for part in (0, 1)]
print(f"{len(encoding.tokens)} tokens in one input: {len(segments[0])} with segment 0 (the question), "
      f"{len(segments[1])} with segment 1 (the chunk)")
print("segment 0:", " ".join(segments[0]))
print("segment 1:", " ".join(segments[1][:22]), "...", " ".join(segments[1][-4:]))

64 tokens in one input: 12 with segment 0 (the question), 52 with segment 1 (the chunk)
segment 0: [CLS] how much time off do new dad ##s get ? [SEP]
segment 1: parental leave policy > leave en ##ti ##tlement primary care ##gi ##vers receive 16 weeks of fully paid leave . secondary care ... s arrival . [SEP]


### 2.3 One number per pair

The model outputs one number per pair: a **logit**. Higher means more relevant. It has no fixed range, and it isn't a probability (a sigmoid would squeeze it into 0 to 1, but that wouldn't make it calibrated for this corpus). The first check below runs the model card's own example and compares it with the scores the card prints. That catches a wrong file, a wrong tokenizer setting or the texts in the wrong order.

In [7]:
def rerank_scores(question: str, texts: list[str], batch_size: int = 16) -> list[float]:
    # One number per (question, text) pair. The model reads the question and the chunk TOGETHER, in one input, and
    # outputs a single logit: higher means more relevant. It isn't a probability and has no fixed range.
    scores = []
    for start in range(0, len(texts), batch_size):
        encodings = pair_tokenizer.encode_batch([(question, text) for text in texts[start:start + batch_size]])
        inputs = {"input_ids": np.array([e.ids for e in encodings], dtype=np.int64),
                  "attention_mask": np.array([e.attention_mask for e in encodings], dtype=np.int64),
                  "token_type_ids": np.array([e.type_ids for e in encodings], dtype=np.int64)}
        scores.extend(float(s) for s in reranker.run(None, inputs)[0][:, 0])
    return scores


# A check that the files load and are used the right way: the model card's own example and its printed scores
CARD_QUESTION = "How many people live in Berlin?"
CARD_PASSAGES = ["Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers.",
                 "Berlin is well known for its museums."]
CARD_SCORES = (8.607138, -4.320078)          # printed on the model card (sentence-transformers, PyTorch)
card = rerank_scores(CARD_QUESTION, CARD_PASSAGES)
print(f"model card example: {card[0]:.4f} and {card[1]:.4f} (the card prints {CARD_SCORES[0]} and {CARD_SCORES[1]})")
# Stop here if the setup is wrong: a wrong file, swapped texts or missing segment ids all change these two numbers
assert abs(card[0] - CARD_SCORES[0]) < 0.001 and abs(card[1] - CARD_SCORES[1]) < 0.001, \
    "the model card's example doesn't reproduce: check the pinned files and that each pair is (question, chunk)"
alone =[rerank_scores(DADS.question, [c.text])[0] for c in chunks[:6]]
batched = rerank_scores(DADS.question, [c.text for c in chunks[:6]])
print(f"one pair at a time vs six in a padded batch: largest difference {max(abs(a - b) for a, b in zip(alone, batched)):.1e}")

model card example: 8.6071 and -4.3201 (the card prints 8.607138 and -4.320078)
one pair at a time vs six in a padded batch: largest difference 0.0e+00


The ONNX export reproduces the card's PyTorch scores to four decimals, and padding a batch changes nothing.

### 2.4 Two ways to score a chunk

For "How much time off do new dads get?", here are the weighted fusion's top 5, each with its rank from dense retrieval, from the hybrid and from the cross-encoder:

In [8]:
def explain(q, base: str = "weighted", n: int = 5) -> None:
    # The hybrid's top n for q, each with its rank and score from dense retrieval, the hybrid and the cross-encoder
    top = [cid for cid, _ in rankings[base][q.id][:n]]
    logits = dict(zip(top, rerank_scores(q.question, [by_cid[cid].text for cid in top])))
    order = {cid: r for r, cid in enumerate(sorted(top, key=lambda c: (-logits[c], c)), start=1)}
    dense_rank = {cid: r for r, (cid, _) in enumerate(rankings["dense"][q.id], start=1)}
    print(f"{q.id}: {q.question}")
    print(f"{'chunk':<44}{'dense':>14}{base:>15}{'cross-encoder':>21}")
    for r, cid in enumerate(top, start=1):
        cosine = dict(rankings["dense"][q.id])[cid]
        print(f"{cid:<44}{f'#{dense_rank[cid]}  {cosine:.3f}':>14}{f'#{r}':>15}{f'#{order[cid]}  {logits[cid]:+7.3f}':>21}"
              f"{'   <- the answer' if cid in q.answer_chunks else ''}")


explain(DADS)

d01-01: How much time off do new dads get?
chunk                                                dense       weighted        cross-encoder
parental-leave-policy#returning-to-work          #1  0.721             #1          #3   -4.651
parental-leave-policy#who-is-eligible            #2  0.672             #2          #2   -4.217
parental-leave-policy#leave-entitlement          #3  0.665             #3          #1   -3.565   <- the answer
remote-work-policy#core-hours                    #6  0.560             #4          #5  -10.777
remote-work-policy#eligibility                   #4  0.631             #5          #4   -9.891


The dense model ranks the three parental-leave chunks within 0.06 of each other, and the answer comes third. The cross-encoder puts the answer first, ahead of the other two leave chunks. It still scores all three below zero, because "dads" never appears and "caregivers" does. **Relevance is relative here, too**: section 6 comes back to that.

## 3. Rerank the hybrid's candidates

The reranker is too slow to read every chunk for every question (section 5 measures it), so it reads only the hybrid's top **N** candidates and puts them in its own order. Everything below N keeps the hybrid's order, so hit@5 and hit@10 stay defined. How large should N be? Larger N gives the reranker more chances to find an answer the hybrid ranked low, and also more chances to promote something wrong.

### 3.1 How many candidates: the pre-registered rule, on dev

The rule, committed in [`evals/preregistration/day-09.md`](../evals/preregistration/day-09.md) before any test-split run: *for each hybrid, the N in {3, 5, 10, 20, 44} that puts a labelled chunk in the top 3 for the most dev queries (16 answerable dev questions and 30 lookups); ties go to the higher dev MRR@10, then to the smaller N.* The sweep below scores dev queries only.

In [9]:
pair_scores = {}     # (query id, chunk id) -> logit. Each pair is scored once and reused by every depth below.


def rerank(q, ranked, n: int) -> list[tuple[str, float]]:
    # The hybrid's top n, put in the cross-encoder's order (ties by chunk id), then the rest of the hybrid's
    # ranking in its own order. Only the first n are ever sent to the model.
    top = [cid for cid, _ in ranked[:n]]
    missing = [cid for cid in top if (q.id, cid) not in pair_scores]
    for cid, score in zip(missing, rerank_scores(q.question, [by_cid[cid].text for cid in missing])):
        pair_scores[q.id, cid] = score
    return sorted(((cid, pair_scores[q.id, cid]) for cid in top), key=lambda kv: (-kv[1], kv[0])) + ranked[n:]


DEV = [q for q in queries if q.split == "dev" and q.answerable]     # the queries we may tune on (16 + 30 lookups)
depth_sweep = {base: {n: measure({q.id: rerank(q, rankings[base][q.id], n) for q in DEV}, DEV) for n in DEPTHS}
               for base in HYBRIDS}
# The rule (pre-registered; dev only): most dev queries in the top 3, then higher dev MRR@10, then the smaller N
DEPTH = {base: max(DEPTHS, key=lambda n: (depth_sweep[base][n]["hit@3"], depth_sweep[base][n]["MRR@10"], -n))
         for base in HYBRIDS}
print(f"{'dev only (46)':<15}" + "".join(f"{f'N = {n}':>16}" for n in DEPTHS))
for base in HYBRIDS:
    print(f"{base:<15}" + "".join(f"{depth_sweep[base][n]['hit@3']:>7} / {depth_sweep[base][n]['MRR@10']:.4f}"
                                  for n in DEPTHS) + f"   -> N = {DEPTH[base]}")
print("(each cell: dev queries with a labelled chunk in the top 3 / dev MRR@10)")
assert DEPTH == PREREGISTERED["depth"], "the rule doesn't give the pre-registered depths: stop and find out why"

RERANKED = {f"{base}+rerank": base for base in HYBRIDS}
for name, base in RERANKED.items():
    rankings[name] = {q.id: rerank(q, rankings[base][q.id], DEPTH[base]) for q in queries}
NAMES = DAY08_NAMES + list(RERANKED)


def rerank_margin(name: str, q) -> float | None:
    # How close the first labelled chunk is to a tie that would move it: its reranker-score gap to a neighbour among
    # the reranked candidates (or its fused-score gap below them), and, if it sits right at the candidate cut-off,
    # the hybrid's gap there (a swap at #N / #N+1 decides whether the reranker sees it at all).
    base = RERANKED[name]
    n, fused, ranked = DEPTH[base], rankings[base][q.id], rankings[name][q.id]
    rank = first_relevant(ranked, q.answer_chunks)
    if rank is None:
        return None
    if rank <= n:
        gaps = [ranked[i - 1][1] - ranked[i][1] for i in (rank - 1, rank) if 0 < i < n]
    else:
        gaps = [fused[i - 1][1] - fused[i][1] for i in (rank - 1, rank) if n < i < len(fused)]
    if first_relevant(fused, q.answer_chunks) in (n, n + 1) and len(fused) > n:
        gaps.append(fused[n - 1][1] - fused[n][1])
    return round(min(gaps), 4) if gaps else None

dev only (46)             N = 3           N = 5          N = 10          N = 20          N = 44
weighted            46 / 0.9855     45 / 0.9837     45 / 0.9837     45 / 0.9826     45 / 0.9826   -> N = 3
RRF                 45 / 0.9674     44 / 0.9620     44 / 0.9609     45 / 0.9826     45 / 0.9826   -> N = 20
(each cell: dev queries with a labelled chunk in the top 3 / dev MRR@10)


For the weighted fusion, **N = 3 is the only depth that keeps every dev query's answer in the top 3**. At every larger N, the reranker reads a chunk from below the top 3 that it likes better than the lost-laptop answer (section 4 shows why) and pushes that answer out. For RRF, whose top 3 is weaker, N = 20 does best: deep enough to reach answers RRF ranked far down.

Note what N = 3 means for the weighted fusion: the reranker reads exactly the three chunks the reader will get, so it **can only change their order**. It can't add an answer to the top 3, and it can't lose one.

### 3.2 All six retrievers, on the same 81 queries

Day 08's four, and the two reranked hybrids (each at its rule's N). The questions are split into dev and test below, because only the test split wasn't used to choose N.

In [10]:
def table(names) -> None:
    print(f"{'':<25}{'n':>3}" + "".join(f"{'hit@' + str(k):>7}" for k in K_VALUES) + f"{'recall@3':>10}{'MRR@10':>8}"
          "   hit@3 rate, 95% interval")
    for kind, selection in SELECTIONS.items():
        for name in names:
            m = measure(rankings[name], selection)
            low, high = wilson(m["hit@3"], m["n"])
            print(f"{kind + ', ' + name:<25}{m['n']:>3}" + "".join(f"{m['hit@' + str(k)]:>7}" for k in K_VALUES)
                  + f"{m['recall@3']:>10.2f}{m['MRR@10']:>8.2f}   {m['hit@3'] / m['n']:.0%} ({low:.0%} to {high:.0%})")


table(NAMES)
print()
for split in ("dev", "test"):
    part = [q for q in SELECTIONS["questions"] if q.split == split]
    print(f"answerable questions, {split} ({len(part)}): " + " | ".join(
        f"{name} {measure(rankings[name], part)['hit@1']}/{measure(rankings[name], part)['hit@3']}" for name in NAMES)
          + "   (first / in the top 3)")

                           n  hit@1  hit@3  hit@5 hit@10  recall@3  MRR@10   hit@3 rate, 95% interval
questions, BM25           31     24     29     29     30      0.94    0.85   94% (79% to 98%)
questions, dense          31     26     29     31     31      0.94    0.90   94% (79% to 98%)
questions, RRF            31     26     30     30     30      0.97    0.90   97% (84% to 99%)
questions, weighted       31     27     31     31     31      1.00    0.92   100% (89% to 100%)
questions, weighted+rerank 31     28     31     31     31      1.00    0.94   100% (89% to 100%)
questions, RRF+rerank     31     28     29     31     31      0.94    0.93   94% (79% to 98%)
lookups, BM25             30     30     30     30     30      1.00    1.00   100% (89% to 100%)
lookups, dense            30     12     19     26     29      0.62    0.56   63% (46% to 78%)
lookups, RRF              30     20     30     30     30      1.00    0.82   100% (89% to 100%)
lookups, weighted         30     30     30 

In the top 3, reranking the weighted fusion changes nothing (it can't, at N = 3). It puts one more question first overall (28 vs 27): two more on dev and one fewer on test. Reranking RRF lifts its lookups to 30 of 30 first (RRF alone: 20), but it ends with fewer questions in the top 3 (29 vs 30).

### 3.3 Query by query, against Day 08's accepted baseline

As on Days 07 and 08, counts hide which queries moved. Each reranked hybrid is compared with its own hybrid **as recorded in `evals/baselines/day-08-retrieval.json`**, pair by pair, with the sign test. Dev and test are kept apart, and **the test split is the pre-registered comparison**. A query whose difference CPU rounding alone could explain (a recorded near tie, under 0.001, right at the cut-off) is set apart, not counted.

In [11]:
def accepted_rank(name: str, qid: str):
    # Day 08's retrievers: the rank recorded in the accepted baseline file. Today's: this run's.
    if name in day08["retrievers"]:
        return day08["queries"][qid][name]["first_relevant"], day08["queries"][qid][name]["margin"]
    e = entry(name, by_qid[qid])
    return e["first_relevant"], e["margin"]


PARTS = {"dev questions": [q for q in SELECTIONS["questions"] if q.split == "dev"],
         "test questions": [q for q in SELECTIONS["questions"] if q.split == "test"],
         "lookups": SELECTIONS["lookups"]}
PAIRED, NEAR = {}, {}


def show_paired(a: str, b: str, k: int, parts=tuple(PARTS)) -> None:
    for part in parts:
        selection = PARTS[part]
        ranks = {name: {q.id: accepted_rank(name, q.id) for q in selection} for name in (a, b)}
        found = {name: {qid for qid, (r, _) in ranks[name].items() if r is not None and r <= k} for name in (a, b)}
        # CPU rounding could explain a difference only if one side is a near tie (< TIE from a neighbour) sitting right
        # at the cut-off (rank k or k + 1), where one swap moves it across: those are set apart, not counted
        near = {qid for qid in found[a] ^ found[b]
                if any(ranks[n][qid][0] in (k, k + 1) and (ranks[n][qid][1] or 1) < TIE for n in (a, b))}
        groups = {"both": found[a] & found[b], f"only {a}": found[a] - found[b] - near,
                  f"only {b}": found[b] - found[a] - near, "neither": {q.id for q in selection} - found[a] - found[b]}
        PAIRED[a, b, k, part] = {label: sorted(ids) for label, ids in groups.items()}
        NEAR[a, b, k, part] = sorted(near)
        p = sign_test(len(groups[f"only {a}"]), len(groups[f"only {b}"]))
        where = "first" if k == 1 else f"in the top {k}"
        print(f"{a} vs {b}, {part} ({len(selection)}), labelled chunk {where}: "
              + " | ".join(f"{label} {len(ids)}" for label, ids in groups.items()) + f" | sign test p = {p:.2g}"
              + (f" | near ties set apart: {', '.join(sorted(near))}" if near else ""))
        for label in (f"only {a}", f"only {b}"):
            if groups[label]:
                print(f"    {label:<22} " + ", ".join(sorted(groups[label])))


for k in (1, 3):
    for base in HYBRIDS:
        show_paired(f"{base}+rerank", base, k)
    print()

weighted+rerank vs weighted, dev questions (16), labelled chunk first: both 13 | only weighted+rerank 2 | only weighted 0 | neither 1 | sign test p = 0.5
    only weighted+rerank   d01-01, d03-04
weighted+rerank vs weighted, test questions (15), labelled chunk first: both 13 | only weighted+rerank 0 | only weighted 1 | neither 1 | sign test p = 1
    only weighted          d07-a02
weighted+rerank vs weighted, lookups (30), labelled chunk first: both 30 | only weighted+rerank 0 | only weighted 0 | neither 0 | sign test p = 1
RRF+rerank vs RRF, dev questions (16), labelled chunk first: both 13 | only RRF+rerank 2 | only RRF 0 | neither 1 | sign test p = 0.5
    only RRF+rerank        d01-01, d03-04
RRF+rerank vs RRF, test questions (15), labelled chunk first: both 12 | only RRF+rerank 1 | only RRF 1 | neither 1 | sign test p = 1
    only RRF+rerank        d07-a07
    only RRF               d07-a02
RRF+rerank vs RRF, lookups (30), labelled chunk first: both 20 | only RRF+rerank 9 | only R

Read the counts:

- **Weighted fusion, reranked, first place:** two gains on dev (parental leave, new dads) and one loss on test ("Do I need a receipt for a USD 40 dinner?"). Neither split shows a difference the sign test can tell from luck (p = 0.5 and p = 1).
- **RRF, reranked:** it wins back first place for 9 lookups that RRF had demoted (p ≈ 0.004), plus a tenth that sits on a near tie (d04-v10, which RRF ranks 1 or 2 depending on rounding). In the top 3, it gains new dads on dev but loses the lost laptop (dev) and wine (test).

So the reranker doesn't simply improve things. It replaces the hybrid's mistakes with its own. Here is each move.

## 4. What moved, and why

In [12]:
# Every query whose rank changed when its hybrid was reranked, with the reason (written after reading each one).
# The assert fails if the set of moves changes, so a new move can't go unexplained.
WHY_MOVED = {
    ("weighted+rerank", "d03-04"): "'how long' meets '16 weeks': the answer chunk now beats Who is eligible",
    ("weighted+rerank", "d01-01"): "all three leave chunks score low (dads vs caregivers), but the answer least low",
    ("weighted+rerank", "d07-a02"): "USD 40 vs 'over USD 25' is arithmetic: Meals edges out the receipt rule",
    ("RRF+rerank", "d03-04"): "'how long' meets '16 weeks': the answer chunk now beats Who is eligible",
    ("RRF+rerank", "d01-01"): "BM25 never returns it, RRF ranked it 15th; inside the top 20 the reranker reads it",
    ("RRF+rerank", "d03-13"): "each chunk scored alone: How to report never mentions a laptop",
    ("RRF+rerank", "d07-a02"): "USD 40 vs 'over USD 25' is arithmetic: Meals edges out the receipt rule",
    ("RRF+rerank", "d07-a07"): "Booking travel answers 'how far in advance'; RRF had the expense policy's Travel first",
    ("RRF+rerank", "d07-a08"): "'wine' never appears; Alcohol does: Meals falls behind three other chunks",
}
WHY_MOVED.update({("RRF+rerank", qid): "an exact value: the chunk that contains it goes first, as with BM25 alone"
                  for qid in ("d04-v07", "d04-v10", "d04-v11", "d04-v12", "d04-v17", "d04-v18", "d04-v20", "d04-v21",
                              "d04-v22", "d04-v24")})
moves = [(name, q) for name, base in RERANKED.items() for q in queries if q.answerable
         and first_relevant(rankings[name][q.id], q.answer_chunks) != first_relevant(rankings[base][q.id], q.answer_chunks)]
assert {(name, q.id) for name, q in moves} == set(WHY_MOVED), "every move needs a reason, and only moves"
print(f"{'reranked':<17}{'query':<9}{'split':<6}{'rank':>9}   why")
for name, q in moves:
    before = first_relevant(rankings[RERANKED[name]][q.id], q.answer_chunks)
    print(f"{name:<17}{q.id:<9}{q.split:<6}{f'{before} -> {first_relevant(rankings[name][q.id], q.answer_chunks)}':>9}   "
          f"{WHY_MOVED[name, q.id]}")

reranked         query    split      rank   why
weighted+rerank  d03-04   dev      2 -> 1   'how long' meets '16 weeks': the answer chunk now beats Who is eligible
weighted+rerank  d01-01   dev      3 -> 1   all three leave chunks score low (dads vs caregivers), but the answer least low
weighted+rerank  d07-a02  test     1 -> 2   USD 40 vs 'over USD 25' is arithmetic: Meals edges out the receipt rule
RRF+rerank       d03-04   dev      2 -> 1   'how long' meets '16 weeks': the answer chunk now beats Who is eligible
RRF+rerank       d03-13   dev      2 -> 5   each chunk scored alone: How to report never mentions a laptop
RRF+rerank       d01-01   dev     15 -> 1   BM25 never returns it, RRF ranked it 15th; inside the top 20 the reranker reads it
RRF+rerank       d04-v07  dev      2 -> 1   an exact value: the chunk that contains it goes first, as with BM25 alone
RRF+rerank       d04-v10  dev      2 -> 1   an exact value: the chunk that contains it goes first, as with BM25 alone
RRF+rerank

Three kinds of reasons, and the last two are the cross-encoder's own limits:

- **It reads the question and the chunk together.** "How long" finds "16 weeks"; "how far in advance" finds the booking rules; an exact value finds the chunk that contains it, which undoes RRF's demotion of BM25's first place (Day 08, section 4.1).
- **It scores each chunk alone.** "How soon must I report a lost laptop?" needs two sections: *What counts as an incident* (a lost laptop is one) and *How to report* (within one hour). The second never mentions a laptop, so the reranker ranks it 5th. Neither retriever nor reranker can join two chunks; Day 11 retrieves surrounding sections.
- **It doesn't reason.** A USD 40 dinner needs a receipt because 40 is more than 25, and wine is alcohol. Neither is a matter of matching words. The reranker scores the right chunk for both very low, below chunks that share more vocabulary.

### 4.1 Every remaining miss

Day 08's 16 misses are unchanged (its four retrievers are). The new ones belong to RRF reranked; the weighted fusion reranked has none. As on Days 07 and 08, the `assert` fails if the set of misses changes.

In [13]:
DIAGNOSES = {(e["retriever"], e["id"]): (e["category"], e["why"]) for e in day08["errors"]}   # Day 08's, unchanged
NEW_DIAGNOSES = {
    ("RRF+rerank", "d03-13"): ("other section", "the reranker scores each chunk alone: How to report (rank 5) holds 'within "
                                                "one hour' but never mentions a laptop; What counts as an incident does"),
    ("RRF+rerank", "d07-a08"): ("vocabulary", "the question says wine, the Meals chunk says Alcohol: the reranker ranks it "
                                              "4th, behind Scope, the travel policy's Purpose and a chunk RRF had 9th"),
}
DIAGNOSES.update(NEW_DIAGNOSES)
misses = [(name, q) for name in NAMES for q in queries if q.answerable and not hit(rankings[name][q.id], q.answer_chunks, TOP_K)]
assert {(name, q.id) for name, q in misses} == set(DIAGNOSES), "every miss needs a diagnosis, and only misses"
print(f"misses in the top 3: " + ", ".join(f"{name} {sum(n == name for n, _ in misses)}" for name in NAMES))
for name, q in misses:
    if name in RERANKED:
        category, why = DIAGNOSES[name, q.id]
        print(f"  {name:<17}{q.id:<9}{q.split:<6}{category:<17}{q.question}\n{'':<49}{why}")

misses in the top 3: BM25 2, dense 13, RRF 1, weighted 0, weighted+rerank 0, RRF+rerank 2
  RRF+rerank       d03-13   dev   other section    How soon must I report a lost laptop?
                                                 the reranker scores each chunk alone: How to report (rank 5) holds 'within one hour' but never mentions a laptop; What counts as an incident does
  RRF+rerank       d07-a08  test  vocabulary       Will Acme pay for wine with a business dinner?
                                                 the question says wine, the Meals chunk says Alcohol: the reranker ranks it 4th, behind Scope, the travel policy's Purpose and a chunk RRF had 9th


## 5. What the second look costs

A reranker runs a model once per candidate, for every query. Measured here, on this machine, over 27 queries from both splits, with the model really called (no cached scores). Only the shape is asserted, with wide margins: 44 candidates take more than twice as long as 3, and 20 more than twice as long as embedding one question.

In [14]:
def timed(work, sample, rounds: int = 1) -> tuple[float, float]:
    samples = []
    for _ in range(rounds):
        for q in sample:
            started = time.perf_counter()
            work(q)
            samples.append((time.perf_counter() - started) * 1000)
    samples.sort()
    return statistics.median(samples), samples[max(0, math.ceil(0.95 * len(samples)) - 1)]


SAMPLE = queries[::3]     # 27 queries, both splits; the model is really called here (no cached scores)
timings = {"BM25 (full ranking)": timed(bm25_search, SAMPLE, 3),
           "dense (embed the question + search)": timed(dense_search, SAMPLE, 3),
           "weighted fusion": timed(lambda q: weighted(rankings["BM25"][q.id], rankings["dense"][q.id], ALPHA), SAMPLE, 3)}
for n in DEPTHS:
    timings[f"rerank {n} candidates"] = timed(
        lambda q: rerank_scores(q.question, [by_cid[cid].text for cid, _ in rankings["weighted"][q.id][:n]]), SAMPLE)
for step, (median, p95) in timings.items():
    print(f"{step:<38} median {median:8.2f} ms   p95 {p95:8.2f} ms")

BM25 (full ranking)                    median     0.01 ms   p95     0.04 ms
dense (embed the question + search)    median    10.42 ms   p95    13.57 ms
weighted fusion                        median     0.02 ms   p95     0.03 ms
rerank 3 candidates                    median    31.64 ms   p95    53.44 ms
rerank 5 candidates                    median    54.80 ms   p95    70.34 ms
rerank 10 candidates                   median   107.92 ms   p95   161.53 ms
rerank 20 candidates                   median   235.18 ms   p95   263.72 ms
rerank 44 candidates                   median   492.95 ms   p95   554.63 ms


Embedding the question, the expensive part of hybrid retrieval, takes about 10 ms. **Reranking 3 candidates takes three to four times that, and 20 candidates more than twenty times.** The cost grows with N, because each pair is a full model run over the question and a chunk of up to 93 tokens; the question can't be encoded once and reused, as it can with a bi-encoder. On a GPU or with a smaller model the numbers shrink, but the shape doesn't. Day 13 measures the whole pipeline.

### 5.1 Every depth, on every split (a description, not a choice)

Now that N is committed, here is every depth on every split, with its cost. Nothing here changes the choice: it shows what other choices would have done.

In [15]:
# Every depth on every split, after the rule was committed: a description, not a choice
print(f"{'':<17}{'dev questions':>16}{'test questions':>16}{'lookups':>12}{'model calls':>13}{'median ms':>11}")
print(f"{'':<17}{'(16) @1  @3':>16}{'(15) @1  @3':>16}{'(30) @1  @3':>12}")
for base in HYBRIDS:
    for n in (0, *DEPTHS):
        ranked = rankings[base] if n == 0 else {q.id: rerank(q, rankings[base][q.id], n) for q in queries}
        cells = [measure(ranked, PARTS[p]) for p in PARTS]
        label = f"{base}" if n == 0 else f"  + rerank {n}"
        cost = "-" if n == 0 else f"{timings[f'rerank {n} candidates'][0]:.0f}"
        print(f"{label:<17}" + "".join(f"{c['hit@1']:>{w - 4}}{c['hit@3']:>4}" for c, w in zip(cells, (16, 16, 12)))
              + f"{n:>13}{cost:>11}" + ("   <- the rule's choice" if n == DEPTH[base] else ""))

                    dev questions  test questions     lookups  model calls  median ms
                      (16) @1  @3     (15) @1  @3 (30) @1  @3
weighted                   13  16          14  15      30  30            0          -
  + rerank 3               15  16          13  15      30  30            3         32   <- the rule's choice
  + rerank 5               15  15          13  15      30  30            5         55
  + rerank 10              15  15          13  14      30  30           10        108


  + rerank 20              15  15          13  14      30  30           20        235


  + rerank 44              15  15          13  14      30  30           44        493
RRF                        13  15          13  15      20  30            0          -
  + rerank 3               14  15          13  15      30  30            3         32
  + rerank 5               14  14          13  15      30  30            5         55
  + rerank 10              14  14          13  14      30  30           10        108
  + rerank 20              15  15          13  14      30  30           20        235   <- the rule's choice
  + rerank 44              15  15          13  14      30  30           44        493


On the test split, deeper reranking of the weighted fusion is never better: from N = 10 it also pushes wine out of the top 3. The dev split pointed the same way. **More candidates means more chances to promote a wrong one, at a higher cost.**

## 6. Can the reranker tell "close" from "answers"?

Day 07 found five unanswerable test questions that still get a cited, wrong answer, because the chunks retrieved for them are close without answering them ("How much annual leave do I get?" retrieves the parental-leave entitlement). Their dense scores sit inside the answerable range, so no threshold on cosine similarity separates them. A cross-encoder reads the question with the chunk, so it might. The test is pre-registered.

### 6.1 The scores

Each question's **best reranker score** among the weighted fusion's top 3 (the three chunks the reader would get):

In [16]:
natural = [q for q in queries if q.kind == "question"]      # gates are for questions, not pasted values


def day07_signals(q) -> tuple[float, tuple[str, ...]]:
    # What Day 07's gate and Day 06's names rule read: dense retrieval's top score, and names no source mentions
    results = rag.retrieve(q.user, q.question)
    return results[0].score, tuple(unknown_names(q.question, assemble(q.question, results).sources))


signal = {q.id: day07_signals(q) for q in natural}
best = {q.id: rankings["weighted+rerank"][q.id][0][1] for q in natural}   # the reranker's best score among the 3

print("best reranker score of each question (weighted fusion's top 3, reranked) | ● answerable  ○ not answerable\n")
print(f"{'score':<14}{'dev: 16 ● and 8 ○':<26}test: 15 ● and 12 ○")
bins = {q.id: math.floor(best[q.id]) for q in natural}        # bins of 1.0
for b in range(max(bins.values()), min(bins.values()) - 1, -1):
    row = ["".join("●" for q in natural if q.split == split and q.answerable and bins[q.id] == b) +
           "".join("○" for q in natural if q.split == split and not q.answerable and bins[q.id] == b)
           for split in ("dev", "test")]
    print(f"{b:+3d} to {b + 1:+3d}   {row[0]:<26}{row[1]}")

best reranker score of each question (weighted fusion's top 3, reranked) | ● answerable  ○ not answerable

score         dev: 16 ● and 8 ○         test: 15 ● and 12 ○
 +9 to +10                             ●
 +8 to  +9   ●●●                       ●
 +7 to  +8   ●●                        ●
 +6 to  +7   ●●●                       ●●
 +5 to  +6                             ●●●
 +4 to  +5   ○                         ●●
 +3 to  +4   ●●                        ○
 +2 to  +3   ●●                        ●
 +1 to  +2   ●●                        
 +0 to  +1   ●                         ●
 -1 to  +0                             ○○○
 -2 to  -1                             ●●
 -3 to  -2                             ○
 -4 to  -3   ●                         
 -5 to  -4   ○                         
 -6 to  -5                             
 -7 to  -6                             
 -8 to  -7                             ●
 -9 to  -8   ○                         ○
-10 to  -9                             ○○○
-11 to -1

The cross-encoder spreads the two groups further apart than cosine similarity did on Day 07: most answerable questions score above 0, and most unanswerable ones below −8. But not all of them: on the test split, several unanswerable questions score between −3 and +4, among the answerable ones.

### 6.2 A gate on the reranker's score

The rule, pre-registered and the same as Day 07's for the dense gate: *the highest value, in steps of 0.1, that refuses no answerable dev question.* The gate refuses a question before any generator runs. The table shows it alone and together with Day 07's measured gate (0.62 on the dense top score) and Day 06's names rule.

In [17]:
# The rule (pre-registered; dev only): the highest gate, in steps of 0.1, that refuses no answerable dev question
weakest = min((q for q in natural if q.split == "dev" and q.answerable), key=lambda q: best[q.id])
GATE = round(math.floor(best[weakest.id] * 10) / 10, 1)
print(f"weakest answerable dev question: {weakest.id} \"{weakest.question}\" at {best[weakest.id]:.4f} -> gate {GATE}")
assert GATE == PREREGISTERED["gate"], "the rule doesn't give the pre-registered gate: stop and find out why"

RULES = {   # each refuses a question before any generator is called
    "Day 07: dense gate 0.62 + names rule": lambda q: signal[q.id][0] < DENSE_GATE or bool(signal[q.id][1]),
    f"reranker gate {GATE}": lambda q: best[q.id] < GATE,
    "both (refuse if either does)": lambda q: (signal[q.id][0] < DENSE_GATE or bool(signal[q.id][1]))
                                              or best[q.id] < GATE}
GATE_TABLE = {}
print(f"\n{'':<38}{'dev: answerable kept':>22}{'not answerable refused':>24}{'test: answerable kept':>23}"
      f"{'not answerable refused':>24}")
for label, refuses in RULES.items():
    cells = []
    for split in ("dev", "test"):
        part = [q for q in natural if q.split == split]
        kept = sum(q.answerable and not refuses(q) for q in part)
        refused = sum(not q.answerable and refuses(q) for q in part)
        GATE_TABLE[label, split] = (kept, sum(q.answerable for q in part), refused, sum(not q.answerable for q in part))
        cells += [f"{kept} of {sum(q.answerable for q in part)}", f"{refused} of {sum(not q.answerable for q in part)}"]
    print(f"{label:<38}{cells[0]:>22}{cells[1]:>24}{cells[2]:>23}{cells[3]:>24}")

weakest answerable dev question: d01-01 "How much time off do new dads get?" at -3.5646 -> gate -3.6

                                        dev: answerable kept  not answerable refused  test: answerable kept  not answerable refused
Day 07: dense gate 0.62 + names rule                16 of 16                  8 of 8               13 of 15                 6 of 12
reranker gate -3.6                                  16 of 16                  7 of 8               14 of 15                 7 of 12
both (refuse if either does)                        16 of 16                  8 of 8               12 of 15                 7 of 12


On dev, the reranker gate keeps every answerable question and refuses 7 of 8 unanswerable ones (all but Google, whose remote-work question meets Acme's remote-work policy). It is set by a single question, new dads at −3.56. On test, it keeps 14 of 15 answerable questions and refuses 7 of 12 unanswerable ones.

### 6.3 The five "similar but wrong" questions

In [18]:
FIVE = ["d07-u01", "d07-u03", "d07-u04", "d07-u11", "d07-u12"]    # still answered with Day 07's measured settings
print(f"{'':<9}{'question':<56}{'dense top':>10}{'reranker':>10}   reranker's first chunk")
for qid in FIVE:
    q = by_qid[qid]
    first = rankings["weighted+rerank"][qid][0][0]
    print(f"{qid:<9}{q.question:<56}{signal[qid][0]:>10.3f}{best[qid]:>+10.2f}   {first}"
          + ("   refused" if best[qid] < GATE else ""))
answerable_test = [q for q in natural if q.split == "test" and q.answerable]
lowest = sorted(answerable_test, key=lambda q: best[q.id])[:3]
print("\nlowest answerable test questions: " + ", ".join(f"{q.id} {best[q.id]:+.2f}" for q in lowest))
refused_five = [qid for qid in FIVE if best[qid] < GATE]
refused_answerable = [q.id for q in answerable_test if best[q.id] < GATE]
SEPARATES = len(refused_five) == len(FIVE) and not refused_answerable
print(f"the gate refuses {len(refused_five)} of the 5 and {len(refused_answerable)} of the 15 answerable test questions"
      f" -> {'separates them' if SEPARATES else 'does not separate them'} (the pre-registered test)")

         question                                                 dense top  reranker   reranker's first chunk
d07-u01  How much annual leave do I get?                              0.680     -2.93   parental-leave-policy#leave-entitlement
d07-u03  How long are emails kept?                                    0.726     -0.48   data-retention-policy#retention-periods
d07-u04  Can I work remotely from another country?                    0.708     -0.37   remote-work-policy#purpose
d07-u11  Who approves pay changes?                                    0.678     -0.62   expense-reimbursement-policy#approval-and-payment
d07-u12  How much does Acme pay for a home internet connection?       0.711     +3.15   remote-work-policy#equipment-and-home-office

lowest answerable test questions: d07-a02 -7.38, d07-a08 -1.83, d07-a01 -1.15
the gate refuses 0 of the 5 and 1 of the 15 answerable test questions -> does not separate them (the pre-registered test)


**The reranker doesn't separate them.** It scores four of the five below zero, much lower than most answerable questions, but the dev rule can't put the gate that high: new dads, an answerable dev question, scores −3.56. On test the overlap is worse. The receipt question (answerable) scores −7.38, below all five, and home internet (unanswerable) scores +3.15, above 5 of the 15 answerable test questions: the remote-work policy's home-office section is about what Acme pays for at home (a desk, a chair, a monitor), just not internet.

A cross-encoder trained on web search judges whether a passage is **about** the question, which is the job it was trained for. Whether the passage **contains the answer** is a different judgement. On this corpus the two overlap for most questions and not for these five. Day 12 measures answers, not passages.

## 7. Context selection: duplicates, diversity and the budget

After ranking comes **selection**: which chunks go to the generator, within a budget. Three questions decide it. Is a chunk a repeat of one already chosen? Should one document fill every slot? And how much text fits?

### 7.1 Counting the budget

Day 06 budgeted 300 words. A language model counts **tokens**, and every tokenizer splits text differently. Counted with the reranker's tokenizer (BERT's WordPiece):

In [19]:
counter = Tokenizer.from_file(str(RERANKER_DIR / "tokenizer.json"))     # no padding, no truncation: just count


def count_tokens(text: str) -> int:
    return len(counter.encode(text, add_special_tokens=False).ids)


tokens = sorted(count_tokens(c.text) for c in chunks)
words = sum(count_words(c.text) for c in chunks)
print(f"tokens per chunk: {tokens[0]} to {tokens[-1]} (median {statistics.median(tokens):.0f}) | all 44 chunks: "
      f"{sum(tokens)} tokens, {words} words: {sum(tokens) / words:.2f} tokens per word")
print(f"Day 06's budget, {MAX_CONTEXT_WORDS} words, is about {MAX_CONTEXT_WORDS * sum(tokens) / words:.0f} of these "
      "tokens; a language model's own tokenizer gives other counts")

tokens per chunk: 21 to 93 (median 44) | all 44 chunks: 1962 tokens, 1621 words: 1.21 tokens per word
Day 06's budget, 300 words, is about 363 of these tokens; a language model's own tokenizer gives other counts


A production pipeline counts with the tokenizer of the model that will read the context, and leaves room for the instructions, the question and the answer. The ratio here (1.21 tokens per word) is this tokenizer's on this corpus, not a general rule.

### 7.2 Selecting the context

`select_context()` walks the ranking best first and fills up to three slots. It skips a chunk whose text repeats one already chosen (the same text, or the same text without its header, which is a copy saved under another title), a chunk from a document that already has its share (if a cap is set), and a chunk that would go over the budget. A skipped chunk **frees its slot** for the next one. Day 06's `assemble()` dropped a repeat and sent fewer chunks.

The pre-registered default is Day 06's: 3 chunks, 300 words, no cap. On the real corpus it selects exactly the top 3 for every answerable query: no chunk is long enough to break the budget, and no text repeats.

In [20]:
def body(c) -> str:
    # A chunk's text without its first line, the "Title > Section" header (Day 03)
    return c.text.split("\n", 1)[1]


def select_context(ranked, lookup, max_chunks: int = TOP_K, budget: int = MAX_CONTEXT_WORDS, count=count_words,
                   per_document: int | None = None, dedupe: bool = True) -> tuple[list[str], list[tuple[str, str]]]:
    # Walk the ranking best first and fill up to max_chunks slots. Skip a repeated text (the same chunk text, or the
    # same text under another title), a chunk from a document that already has per_document chunks, and a chunk
    # that would go over the budget. A skipped chunk frees its slot for the next one.
    chosen, dropped, seen, per_doc, used = [], [], set(), Counter(), 0
    for cid, _ in ranked:
        c = lookup[cid]
        same_text = {c.chunk_hash, hashlib.sha256(body(c).encode("utf-8")).hexdigest()}
        if dedupe and same_text & seen:
            dropped.append((cid, "repeated text"))
        elif per_document and per_doc[c.doc_id] >= per_document:
            dropped.append((cid, "document cap"))
        elif used + count(c.text) > budget:
            dropped.append((cid, "over budget"))
        else:
            chosen.append(cid)
            seen |= same_text
            per_doc[c.doc_id] += 1
            used += count(c.text)
            if len(chosen) == max_chunks:
                break
    return chosen, dropped


PRIMARY = "weighted+rerank"
ANSWERABLE = SELECTIONS["questions"] + SELECTIONS["lookups"]
default = {q.id: select_context(rankings[PRIMARY][q.id], by_cid)[0] for q in ANSWERABLE}
print(f"default selection (3 chunks, {MAX_CONTEXT_WORDS} words, no cap): the top 3 for "
      f"{sum(default[q.id] == [c for c, _ in rankings[PRIMARY][q.id][:3]] for q in ANSWERABLE)} of {len(ANSWERABLE)} "
      f"answerable queries | repeated texts in the real corpus: "
      f"{len(chunks) - len({hashlib.sha256(body(c).encode('utf-8')).hexdigest() for c in chunks})}")

default selection (3 chunks, 300 words, no cap): the top 3 for 61 of 61 answerable queries | repeated texts in the real corpus: 0


### 7.3 Diversity and the budget, on every answerable query

A **per-document cap** (at most n chunks from one document) is a simple form of source diversity. The pre-registration adopts a setting only if it gains something on dev and loses nothing, so the cap isn't adopted; here it is on every split, as a description, along with smaller budgets (counted in tokens).

In [21]:
def in_context(selection_of, part) -> int:
    return sum(bool(set(selection_of[q.id]) & set(q.answer_chunks)) for q in PARTS[part])


GRID = {}
print(f"{'per document':<14}{'budget':<12}{'dev q (16)':>11}{'test q (15)':>12}{'lookups (30)':>13}{'median tokens':>15}")
for cap in (None, 2, 1):
    for budget, count, label in [(64, count_tokens, "64 tokens"), (128, count_tokens, "128 tokens"),
                                 (256, count_tokens, "256 tokens"), (MAX_CONTEXT_WORDS, count_words, "300 words")]:
        sel = {q.id: select_context(rankings[PRIMARY][q.id], by_cid, budget=budget, count=count, per_document=cap)[0]
               for q in ANSWERABLE}
        GRID[cap, label] = [in_context(sel, p) for p in PARTS]
        median_tokens = statistics.median(sum(count_tokens(by_cid[c].text) for c in sel[q.id]) for q in ANSWERABLE)
        print(f"{str(cap or 'no cap'):<14}{label:<12}" + "".join(f"{v:>{w}}" for v, w in zip(GRID[cap, label], (11, 12, 13)))
              + f"{median_tokens:>15.0f}" + ("   <- Day 06's budget, today's default" if cap is None and count is count_words else ""))
for cap in (2, 1):
    lost = [q.id for q in ANSWERABLE if set(default[q.id]) & set(q.answer_chunks)
            and not set(select_context(rankings[PRIMARY][q.id], by_cid, per_document=cap)[0]) & set(q.answer_chunks)]
    print(f"a cap of {cap} per document loses: {', '.join(lost) or 'nothing'}")

per document  budget       dev q (16) test q (15) lookups (30)  median tokens


no cap        64 tokens            12          12           22             60
no cap        128 tokens           15          15           30            121
no cap        256 tokens           16          15           30            135


no cap        300 words            16          15           30            135   <- Day 06's budget, today's default


2             64 tokens            12          12           22             60
2             128 tokens           15          15           30            122
2             256 tokens           16          15           30            138


2             300 words            16          15           30            138


1             64 tokens            12          11           22             58
1             128 tokens           15          13           30            120
1             256 tokens           15          13           30            140
1             300 words            15          13           30            140
a cap of 2 per document loses: nothing
a cap of 1 per document loses: d03-13, d07-a02, d07-a08


A cap of 2 changes nothing. **A cap of 1 loses three answers**: lost laptop, receipt and wine all need a second chunk from the same document in the top 3. On this labelled set every answer is in one chunk, so diversity can only cost. A cap pays off when a question needs several documents ("compare the travel and expense meal limits"), and none of the 81 queries does. Budgets push the other way: at 128 tokens one dev answer falls out, and at 64 tokens fifteen do (4 dev questions, 3 test questions and 8 lookups).

### 7.4 Duplicates, and a stale copy

The real corpus has no repeated text, so here is a temporary copy of it with two extra files (Day 05's `corpus_after()`; `data/sample/` is untouched): the remote-work policy **saved again under another title**, and **an older version** with a different home-office allowance that nobody deleted. Both get permissions, because ingestion would otherwise quarantine them (Day 02).

In [22]:
HOME_OFFICE = by_qid["d03-07"]                 # "How much is the home office allowance?" (dev)
POLICY = "acme-handbook/remote-work-policy.md"


def add_copies(sample: Path) -> None:
    # Two extra files on a temporary copy of data/sample: the same policy saved again under another title, and an
    # older version (a different allowance) that nobody deleted. Both get permissions, or ingestion quarantines them.
    text = (sample / POLICY).read_text(encoding="utf-8")
    (sample / "acme-handbook/remote-work-policy-copy.md").write_text(
        text.replace("title: Remote Work Policy", "title: Remote Work Policy (copy)"), encoding="utf-8")
    old = text.replace("title: Remote Work Policy", "title: Remote Work Policy 2025").replace("USD 500", "USD 400")
    (sample / "acme-handbook/remote-work-policy-2025.md").write_text(old, encoding="utf-8")
    acl = json.loads((sample / "access-control.json").read_text(encoding="utf-8"))
    for name in ("remote-work-policy-copy.md", "remote-work-policy-2025.md"):
        acl["permissions"][f"acme-handbook/{name}"] = acl["permissions"][POLICY]
    (sample / "access-control.json").write_text(json.dumps(acl, indent=2), encoding="utf-8")


copies = corpus_after(REPO_ROOT, add_copies)
copy_lookup = {c.chunk_id: c for c in copies}
copy_index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
sync(copy_index, copies, embedder.embed)
copy_bm25 = BM25Index({c.chunk_id: c.text for c in copies if set(c.allowed_groups) & DIRECTORY[HOME_OFFICE.user]})
copy_dense = [(r.id, r.score) for r in rag.retrieve(HOME_OFFICE.user, HOME_OFFICE.question, len(copies), from_index=copy_index)]
copy_hybrid = weighted([(h.id, h.score) for h in copy_bm25.search(HOME_OFFICE.question, len(copies))], copy_dense, ALPHA)
top = [cid for cid, _ in copy_hybrid[:6]]
copy_reranked = sorted(zip(top, rerank_scores(HOME_OFFICE.question, [copy_lookup[c].text for c in top])),
                       key=lambda kv: (-kv[1], kv[0])) + copy_hybrid[6:]
print(f"{HOME_OFFICE.question} ({len(copies)} chunks now; the hybrid's top 6, reranked)")
for rank, (cid, score) in enumerate(copy_reranked[:6], start=1):
    allowance = next((v for v in ("USD 500", "USD 400") if v in copy_lookup[cid].text), "")
    print(f"  #{rank} {score:+7.3f}  {cid:<52}{allowance}")
for dedupe in (False, True):
    chosen, dropped = select_context(copy_reranked, copy_lookup, dedupe=dedupe)
    values = sorted({v for c in chosen for v in ("USD 500", "USD 400") if v in copy_lookup[c].text})
    print(f"\ncontext {'with' if dedupe else 'without'} dedupe: {', '.join(chosen)}\n  allowances in it: "
          f"{' and '.join(values)}" + (f" | dropped: {', '.join(f'{c} ({why})' for c, why in dropped)}" if dropped else ""))
exact = sum(a.chunk_hash == b.chunk_hash for a in copies for b in copies if a.chunk_id < b.chunk_id)
print(f"\nchunk pairs with identical text (Day 06's check): {exact} | identical text without the header: "
      f"{sum(body(a) == body(b) for a in copies for b in copies if a.chunk_id < b.chunk_id)}")

How much is the home office allowance? (54 chunks now; the hybrid's top 6, reranked)
  #1  +6.901  remote-work-policy#equipment-and-home-office        USD 500
  #2  +6.864  remote-work-policy-2025#equipment-and-home-office   USD 400
  #3  +6.845  remote-work-policy-copy#equipment-and-home-office   USD 500
  #4  -8.113  expense-reimbursement-policy#approval-and-payment   
  #5  -8.576  expense-reimbursement-policy#meals                  
  #6 -10.709  travel-policy#daily-limits-by-city                  

context without dedupe: remote-work-policy#equipment-and-home-office, remote-work-policy-2025#equipment-and-home-office, remote-work-policy-copy#equipment-and-home-office
  allowances in it: USD 400 and USD 500

context with dedupe: remote-work-policy#equipment-and-home-office, remote-work-policy-2025#equipment-and-home-office, expense-reimbursement-policy#approval-and-payment
  allowances in it: USD 400 and USD 500 | dropped: remote-work-policy-copy#equipment-and-home-office (repeated 

The reranker scores the three copies of the home-office section almost equally (+6.90, +6.86, +6.85), and they fill the top 3.

- **Without dedupe**, all three slots go to the same section, and the context holds two different allowances: USD 500 and USD 400.
- **With dedupe**, the exact copy is dropped (its text matches once the header is removed; Day 06's check, on the full text, would miss it) and its slot goes to the next chunk. (The 13 matching pairs are the copy's five sections, each matching the original, and four of the old version's five sections, each matching both.) **The stale version stays**: its text differs by one number, and that number is the problem.

Deduplication saves budget. It can't decide which of two versions is current: that belongs in the data lifecycle, retiring the old version at the source (Day 05's sync and deletes). Day 14 covers freshness.

## 8. Permissions: a reranker promotes what it is given

Every input to the rerankers here was permission-filtered before fusion, so nothing a user may not read can reach them. That is checked first. Then Day 08's mistake again, one shared BM25 index without the filter, now followed by the reranker:

In [23]:
leaks = [(name, q.id) for name in rankings for q in queries
         if set(q.restricted_chunks) & {cid for cid, _ in rankings[name][q.id][:10]}]
print(f"restricted chunks in any top 10 (all {len(rankings)} retrievers, {len(queries)} queries): {len(leaks)}")

shared_bm25 = BM25Index({c.chunk_id: c.text for c in chunks})        # Day 08's mistake: one index, no permission filter
readable = {u: {c.chunk_id for c in chunks if set(c.allowed_groups) & g} for u, g in DIRECTORY.items()}
PROMOTED = []
print(f"\nwith a shared, unfiltered BM25 index (dense still filtered), RRF and then the reranker (top {DEPTH['RRF']}):")
print(f"  {'':<9}{'':<7}{'question':<60}{'restricted chunk':>16}")
for q in queries:
    leaky = [(h.id, h.score) for h in shared_bm25.search(q.question, len(chunks))]
    fused = rrf([leaky, rankings["dense"][q.id]])
    bad = [rank for rank, (cid, _) in enumerate(fused, start=1) if cid not in readable[q.user]]
    if bad and bad[0] <= DEPTH["RRF"]:
        top = [cid for cid, _ in fused[:DEPTH["RRF"]]]
        order = sorted(zip(top, rerank_scores(q.question, [by_cid[c].text for c in top])), key=lambda kv: (-kv[1], kv[0]))
        after = next(r for r, (cid, _) in enumerate(order, start=1) if cid not in readable[q.user])
        PROMOTED.append((q.id, bad[0], after))
        print(f"  {q.id:<9}{q.user:<7}{q.question:<60}{f'RRF #{bad[0]} -> reranked #{after}':>27}")

restricted chunks in any top 10 (all 6 retrievers, 81 queries): 0

with a shared, unfiltered BM25 index (dense still filtered), RRF and then the reranker (top 20):
                  question                                                    restricted chunk


  d06-02   alice  When do pay rises take effect?                                    RRF #4 -> reranked #1


  d06-04   alice  Who is the CEO of Acme?                                          RRF #15 -> reranked #5


  d07-a06  alice  Where do I send a customer's request to delete their data?      RRF #17 -> reranked #20


  d07-a08  alice  Will Acme pay for wine with a business dinner?                   RRF #18 -> reranked #7


  d07-u01  alice  How much annual leave do I get?                                  RRF #5 -> reranked #11


  d07-u02  alice  How many sick days can I take?                                   RRF #8 -> reranked #19


  d07-u11  alice  Who approves pay changes?                                         RRF #6 -> reranked #1


  d07-u12  alice  How much does Acme pay for a home internet connection?          RRF #17 -> reranked #13


Day 08's leak ranked low, and a low rank looked almost harmless. **The reranker undoes that.** For "When do pay rises take effect?" and "Who approves pay changes?", alice's managers-only pay guide goes from RRF #4 and #6 to **#1**. The guide is the most relevant chunk in the collection for both questions; it just isn't hers to read. A relevance model does its job on whatever it is given. Filter in every retriever before fusion, rerank only what passed, and keep Day 06's re-check before generation.

## 9. Decisions

Both decisions were written down, with what each needs, before the test split was scored (`evals/preregistration/day-09.md`):

In [24]:
test_q = PARTS["test questions"]
firsts = {name: measure(rankings[name], test_q)["hit@1"] for name in ("weighted", "weighted+rerank")}
in_top3 = measure(rankings["weighted+rerank"], test_q)["hit@3"]
D1 = firsts["weighted+rerank"] >= firsts["weighted"] and in_top3 == len(test_q)
print(f"D1, reranking in the retriever: test questions first {firsts['weighted+rerank']} (weighted alone "
      f"{firsts['weighted']}), in the top 3 {in_top3} of {len(test_q)} -> "
      + ("adopted: later days build on the weighted fusion, then rerank its top 3" if D1 else
         "not adopted: the weighted fusion stays the default; the reranker stays optional"))

day07_lets_through = [q for q in natural if q.split == "test" and not q.answerable and not RULES[
    "Day 07: dense gate 0.62 + names rule"](q)]
caught = [q.id for q in day07_lets_through if best[q.id] < GATE]
D2 = not refused_answerable and bool(caught)
print(f"D2, the reranker gate: refuses {len(refused_answerable)} of the 15 answerable test questions and {len(caught)} of "
      f"the {len(day07_lets_through)} unanswerable test questions Day 07's gate and names rule let through -> "
      + ("adopted (recorded for later days, on top of Day 07's measured gate)" if D2 else "recorded as measured, not adopted"))
print(f"   refused answerable: {', '.join(refused_answerable) or 'none'} | caught: "
      + (", ".join(f"{qid} ({by_qid[qid].question})" for qid in caught) or "none"))

D1, reranking in the retriever: test questions first 13 (weighted alone 14), in the top 3 15 of 15 -> not adopted: the weighted fusion stays the default; the reranker stays optional
D2, the reranker gate: refuses 1 of the 15 answerable test questions and 1 of the 6 unanswerable test questions Day 07's gate and names rule let through -> recorded as measured, not adopted
   refused answerable: d07-a02 | caught: d07-u05 (How do I order a new company phone?)


**Neither setting is adopted.** The weighted fusion stays the retriever later days build on, and the reranker stays an option, recorded in the baseline with its numbers. The test split decided this by a single question in each case, the receipt question (p = 1 for the ranking), so "not adopted" doesn't mean "worse". It means the evidence didn't clear the bar set in advance, and the cost (30 to 40 ms per query on this machine) is real.

One observation that wasn't planned, so it decides nothing: **as a replacement** for Day 07's gate and names rule (not added on top), the reranker gate kept more answerable test questions (14 vs 13) and refused more unanswerable ones (7 vs 6). Those are differences of one question, on a split now used three times. Testing it properly needs new questions, written and committed before anything scores them (`questions-v2`, `evals/README.md`, rule 4).

**What the v1 test split can still show.** Its questions and every earlier result on it were known before today's rules were written, and the five "similar but wrong" questions were picked on it (by Day 07). It was fair for today's choices of N and gate, because those were fixed without it. It isn't clean evidence for the whole pipeline: the fusion weight under the reranker was chosen after test results had been seen, and the five are a known failure set, not a sample. Today used the split to make two decisions, so under rule 4 its questions now count as used: a later decision that needs held-out evidence needs new test questions.

## 10. Save the run, compare with the accepted baseline

The run goes to `outputs/day-09/retrieval-baseline.json` in Day 08's format, with six retrievers: Day 08's four (unchanged) and the two reranked hybrids. It also records the gate and both decisions. The accepted copy is committed as [`evals/baselines/day-09-retrieval.json`](../evals/baselines/day-09-retrieval.json).

**A reranked ranking's near-tie margin** has two parts. One is its own score gap: between reranker scores among the candidates, or between fused scores below them. The other applies only if the labelled chunk sits right at the cut-off (the hybrid's rank N or N + 1): the hybrid's gap there, because a swap at that point decides whether the reranker reads the chunk at all.

In [25]:
for name, base in RERANKED.items():
    SETTINGS[name] = {"code": "rerank() in the Day 09 notebook", "reranks": base, "depth": DEPTH[base],
                      "depth_rule": "most dev queries in the top 3, then the higher dev MRR@10, then the smaller N",
                      "model": RERANKER["repo"], "revision": RERANKER["revision"],
                      "converted_from": RERANKER["converted_from"],
                      "input": "question, then the chunk text with its Title > Section header; 512 tokens at most",
                      "score": "the model's logit; ties by chunk id", "after_the_depth": "the hybrid's own order"}

baseline = {
    "name": "Day 09 retrieval baseline",
    "dataset": DATASET,
    "retrievers": {name: SETTINGS[name] for name in NAMES},
    "top_k": TOP_K,
    "metrics": metrics_of(NAMES),
    "queries": {q.id: {name: entry(name, q) for name in NAMES} for q in queries},
    "errors": [{"retriever": name, "id": q.id, "first_relevant": first_relevant(rankings[name][q.id], q.answer_chunks),
                "category": DIAGNOSES[name, q.id][0], "why": DIAGNOSES[name, q.id][1]} for name, q in misses],
    "thresholds": {
        "preregistration": PREREGISTRATION.relative_to(REPO_ROOT).as_posix(),
        "reranker_gate": {"rule": "the highest value (in steps of 0.1) that refuses no answerable dev question",
                          "input": "the best reranker score among the weighted fusion's top 3", "value": GATE,
                          "outcomes": {f"{label}, {split}": dict(zip(("answerable kept", "answerable", "unanswerable refused",
                                                                      "unanswerable"), counts))
                                       for (label, split), counts in GATE_TABLE.items()}},
        "day07_measured": "see evals/baselines/day-07-retrieval.json (dense gate 0.62, reader 0.63)",
        "decisions": {"D1 rerank the weighted fusion's top 3": "adopted" if D1 else "not adopted",
                      "D2 reranker gate": "adopted" if D2 else "not adopted"}},
}
OUT_DIR.mkdir(parents=True, exist_ok=True)
run_file = OUT_DIR / "retrieval-baseline.json"
run_file.write_text(json.dumps(baseline, indent=1, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
print(f"this run: {run_file.relative_to(REPO_ROOT).as_posix()} ({len(baseline['queries'])} queries x {len(NAMES)} "
      f"retrievers, {len(baseline['errors'])} errors)")
if BASELINE_FILE.is_file():
    changes, ties = compare(baseline, json.loads(BASELINE_FILE.read_text(encoding="utf-8")))
    print(f"compared with {BASELINE_FILE.relative_to(REPO_ROOT).as_posix()}: "
          + (f"{len(changes)} changes" + "".join(f"\n  {c}" for c in changes) if changes else
             f"no changes ({len(queries)} queries x {len(NAMES)} retrievers and their metrics)"))
    for t in ties:
        print(f"  near tie, not a change: {t}")
else:
    changes, ties = None, []
    print(f"no accepted baseline yet: review {run_file.name}, then copy it to "
          f"{BASELINE_FILE.relative_to(REPO_ROOT).as_posix()} and commit it")
fragile = sorted((e["margin"], qid, name) for qid, per in baseline["queries"].items()
                 for name, e in per.items() if name in RERANKED and e["margin"] is not None)[:3]
print("closest calls among the reranked: " + ", ".join(f"{qid} {name} ({m})" for m, qid, name in fragile))

this run: outputs/day-09/retrieval-baseline.json (81 queries x 6 retrievers, 18 errors)
compared with evals/baselines/day-09-retrieval.json: no changes (81 queries x 6 retrievers and their metrics)
closest calls among the reranked: d07-a08 weighted+rerank (0.0424), d03-13 weighted+rerank (0.0453), d01-01 weighted+rerank (0.0552)


No reranked rank sits near a tie. The closest calls are at the weighted fusion's cut-off, where its #3 and #4 are about 0.04 apart (wine and the lost laptop); the reranker's own scores are further apart still. The near ties that remain are the hybrids' own, carried over from Day 08 (RRF's d03-13 and d04-v10).

## 11. Sanity checks

Each check pins a result quoted in this notebook or in LEARN.md. Scores are compared with a tolerance; ids, ranks and counts exactly. Timings are checked only for their shape.

In [26]:
def close(x: float, expected: float, tol: float = 0.01) -> bool:
    return abs(x - expected) <= tol


def counts(name: str, kind: str) -> tuple:
    m = measure(rankings[name], SELECTIONS[kind])
    return m["hit@1"], m["hit@3"], m["hit@5"], m["hit@10"]


def rank(name: str, qid: str):
    return first_relevant(rankings[name][qid], by_qid[qid].answer_chunks)


# The same inputs as Day 08 (section 1)
assert problems == [] and changes08 == [] and same_top3 and len(queries) == 81 and len(chunks) == 44

# The model (section 2): the pinned files, the model card's own numbers, padding changes nothing
assert same and [i.name for i in inputs_] == ["input_ids", "attention_mask", "token_type_ids"]
assert close(card[0], CARD_SCORES[0], 0.001) and close(card[1], CARD_SCORES[1], 0.001)
assert max(abs(a - b) for a, b in zip(alone, batched)) < 1e-4
assert len(encoding.tokens) == 64 and len(segments[0]) == 12 and segments[0][6:9] == ["new", "dad", "##s"]
assert rerank(DADS, rankings["weighted"][DADS.id], 5)[0][0] == ENTITLEMENT.chunk_id
assert rank("dense", DADS.id) == 3 and pair_scores[DADS.id, ENTITLEMENT.chunk_id] < 0      # first, but still negative

# The depth rule on dev (section 3) and the six retrievers
assert DEPTH == {"weighted": 3, "RRF": 20}
assert all(rankings[name][q.id][DEPTH[base]:] == rankings[base][q.id][DEPTH[base]:]      # below N: the hybrid's order,
           and len(rankings[name][q.id]) == len(rankings[base][q.id])                    # and nothing dropped
           for name, base in RERANKED.items() for q in queries)
assert [depth_sweep["weighted"][n]["hit@3"] for n in DEPTHS] == [46, 45, 45, 45, 45]
assert [depth_sweep["RRF"][n]["hit@3"] for n in DEPTHS] == [45, 44, 44, 45, 45]
assert counts("weighted+rerank", "questions") == (28, 31, 31, 31) and counts("weighted+rerank", "lookups") == (30, 30, 30, 30)
assert counts("RRF+rerank", "questions") == (28, 29, 31, 31) and counts("RRF+rerank", "lookups") == (30, 30, 30, 30)
assert counts("weighted", "questions") == (27, 31, 31, 31) and counts("RRF", "lookups") == (20, 30, 30, 30)
assert [measure(rankings[n], PARTS["test questions"])["hit@1"] for n in NAMES] == [11, 13, 13, 14, 13, 13]
assert [measure(rankings[n], PARTS["test questions"])["hit@3"] for n in NAMES] == [14, 14, 15, 15, 15, 14]
assert [measure(rankings[n], PARTS["dev questions"])["hit@1"] for n in ("weighted", "weighted+rerank", "RRF+rerank")] == [13, 15, 15]
assert PAIRED["weighted+rerank", "weighted", 1, "dev questions"]["only weighted+rerank"] == ["d01-01", "d03-04"]
assert PAIRED["weighted+rerank", "weighted", 1, "test questions"]["only weighted"] == ["d07-a02"]
assert not PAIRED["weighted+rerank", "weighted", 1, "test questions"]["only weighted+rerank"]
assert all(not PAIRED["weighted+rerank", "weighted", 3, p]["only weighted"] for p in PARTS)    # N = 3 can't lose a top 3
assert PAIRED["RRF+rerank", "RRF", 3, "dev questions"]["only RRF+rerank"] == ["d01-01"]
assert PAIRED["RRF+rerank", "RRF", 3, "dev questions"]["only RRF"] == ["d03-13"]
assert PAIRED["RRF+rerank", "RRF", 3, "test questions"]["only RRF"] == ["d07-a08"]
assert len(PAIRED["RRF+rerank", "RRF", 1, "lookups"]["only RRF+rerank"]) == 9 and not PAIRED["RRF+rerank", "RRF", 1, "lookups"]["only RRF"]
assert set(NEAR["RRF+rerank", "RRF", 1, "lookups"]) <= {"d04-v10"}     # RRF's own near tie (Day 08): 1 or 2 on a CPU
assert sign_test(9, 0) < 0.005 and sign_test(2, 0) == 0.5

# What moved, and every miss (section 4)
assert len(moves) == 19 and rank("RRF+rerank", "d01-01") == 1 and rank("RRF", "d01-01") == 15
assert rank("RRF+rerank", "d03-13") == 5 and rank("RRF+rerank", "d07-a08") == 4 and rank("weighted+rerank", "d07-a02") == 2
assert len(misses) == 18 and [(n, q.id) for n, q in misses if n in RERANKED] == [("RRF+rerank", "d03-13"), ("RRF+rerank", "d07-a08")]

# Cost (section 5): the shape only, with wide margins, because timings depend on the machine and on what else is running
assert timings["rerank 20 candidates"][0] > 2 * timings["dense (embed the question + search)"][0]
assert timings["rerank 44 candidates"][0] > 2 * timings["rerank 3 candidates"][0]

# The gate (section 6)
assert GATE == -3.6 and weakest.id == "d01-01" and close(best["d01-01"], -3.5646, 0.001)
assert GATE_TABLE["Day 07: dense gate 0.62 + names rule", "test"] == (13, 15, 6, 12)
assert GATE_TABLE[f"reranker gate {GATE}", "dev"] == (16, 16, 7, 8) and GATE_TABLE[f"reranker gate {GATE}", "test"] == (14, 15, 7, 12)
assert GATE_TABLE["both (refuse if either does)", "test"] == (12, 15, 7, 12)
assert refused_five == [] and refused_answerable == ["d07-a02"] and not SEPARATES
assert all(-3.0 < best[qid] < 3.5 for qid in FIVE) and best["d07-u12"] > 3 and close(best["d07-a02"], -7.38, 0.01)
assert min(abs(best[q.id] - GATE) for q in natural) > 0.03          # no question sits on the gate

# Context selection (section 7)
assert (tokens[0], tokens[-1], sum(tokens), words) == (21, 93, 1962, 1621)
assert all(default[q.id] == [c for c, _ in rankings[PRIMARY][q.id][:3]] for q in ANSWERABLE)
assert GRID[None, "300 words"] == [16, 15, 30] and GRID[2, "300 words"] == [16, 15, 30] and GRID[1, "300 words"] == [15, 13, 30]
assert GRID[None, "64 tokens"] == [12, 12, 22] and GRID[None, "128 tokens"] == [15, 15, 30]
assert len(copies) == 54 and exact == 0
assert [c for c, _ in copy_reranked[:3]] == ["remote-work-policy#equipment-and-home-office",
                                             "remote-work-policy-2025#equipment-and-home-office",
                                             "remote-work-policy-copy#equipment-and-home-office"]
assert select_context(copy_reranked, copy_lookup)[1] == [("remote-work-policy-copy#equipment-and-home-office", "repeated text")]

# Permissions (section 8)
assert fragile[0][0] > 0.04 and fragile[0][1] == "d07-a08"     # logits are far apart: no reranked near tie
assert leaks == [] and len(PROMOTED) == 8 and ("d06-02", 4, 1) in PROMOTED and ("d07-u11", 6, 1) in PROMOTED

# Decisions (section 9) and the baseline (section 10)
assert not D1 and not D2 and caught == ["d07-u05"]
saved = json.loads(run_file.read_text(encoding="utf-8"))
assert saved["dataset"] == day08["dataset"] and len(saved["errors"]) == 18
assert all(saved["queries"][qid][n]["first_relevant"] == day08["queries"][qid][n]["first_relevant"] and
           saved["queries"][qid][n]["top3"] == day08["queries"][qid][n]["top3"] for qid in by_qid for n in DAY08_NAMES)
assert changes is None or changes == [], changes            # near ties (CPU rounding) are allowed
if BASELINE_FILE.is_file():                                  # and the same top 3 lists as the accepted run
    accepted = json.loads(BASELINE_FILE.read_text(encoding="utf-8"))
    assert all(saved["queries"][qid][n]["top3"] == accepted["queries"][qid][n]["top3"] for qid in by_qid for n in NAMES)

print("All Day 09 sanity checks passed.")

All Day 09 sanity checks passed.


## 12. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing anything. The v1 test split has now been used to decide, so treat it as dev for anything you choose.

1. **Rerank the text without its header.** Pass `body(by_cid[cid])` instead of the full text in `rerank()`. On dev it put 12 questions first instead of 15. Why does a cross-encoder benefit from `Title > Section` when Day 04's embedding model didn't?
2. **The original repository's ONNX file.** `cross-encoder/ms-marco-MiniLM-L6-v2` now ships its own `onnx/model.onnx` (91,011,230 bytes, SHA-256 `5d3e70fd0c9ff14b9b5169a51e957b7a9c74897afd0a35ce4bd318150c1d4d4a` at commit `233902d25c440f23af6f7d6e94d2946bac0bee0a`, read from the model page on 2026-10-26; its `tokenizer.json` is the same file as this one). Pin it in `RERANKER`, rerun, and compare: does the model-card check still pass, and does any rank move?
3. **Blend instead of replace.** Rank the top N by `λ × (min-max of the reranker scores) + (1 − λ) × (the fused score)`. Choose λ on dev with Day 08's rule. Does the receipt question get its first place back? What does the blend cost you on new dads?
4. **A question that needs two documents.** Add a query such as "What are the meal limits in the expense policy and the travel policy?" and compare the context with no cap and with a cap of 1 per document. When does diversity earn its place?
5. **Swap the gate.** Run Day 06's `Pipeline` with the reranker gate in place of the dense one: abstain when the best reranker score is below −3.6. Grade the 51 questions with `outcome()` (copy it from Day 07's notebook, section 5; it isn't in `src/`). Which outcomes change, and why can't the v1 test split settle the question any more?